In [ ]:
import os
import requests
import json
import toml
from typing import Optional

# Define the default TOML file path
CONFIG_FILE = "config.toml"

def load_linkedin_token() -> Optional[str]:
    """
    Loads the LinkedIn Access Token from a TOML file or environment variables.
    TOML structure expected: LINKEDIN_ACCESS_TOKEN = "..."
    """
    token = None
    
    # 1. Try to load from the TOML configuration file
    try:
        if os.path.exists(CONFIG_FILE):
            config = toml.load(CONFIG_FILE)
            # Look for the key directly at the root level, as requested
            token = config.get("LINKEDIN_ACCESS_TOKEN") 
            if token:
                print(f"Token loaded from {CONFIG_FILE}")
                return token
    except Exception as e:
        # Note: This warning is useful if the file exists but is malformed
        print(f"Warning: Could not read or parse {CONFIG_FILE}. Falling back to environment variables. Error: {e}")

    # 2. Fall back to environment variables
    token = os.environ.get("LINKEDIN_ACCESS_TOKEN")
    if token:
        print("Token loaded from environment variable LINKEDIN_ACCESS_TOKEN")
        return token

    return None


def post_linkedin_company_update_with_toml(text: str, organization_urn: str) -> Optional[str]:
    """
    Post `text` as a simple text update to a LinkedIn Company Page, 
    loading the token from config.toml or environment variables.

    Args:
        text (str): The content of the post.
        organization_urn (str): The LinkedIn URN of the company page 
                                (e.g., "urn:li:organization:1234567").

    Returns:
        Optional[str]: The created share URN (e.g., "urn:li:share:...") on success, 
                       None on failure.
    """
    
    # Load the access token using the new helper function
    LINKEDIN_ACCESS_TOKEN = load_linkedin_token()

    if not LINKEDIN_ACCESS_TOKEN:
        raise RuntimeError(f"Missing LinkedIn credentials. Ensure 'LINKEDIN_ACCESS_TOKEN' is set in {CONFIG_FILE} or in the environment.")

    # Define API endpoint and headers
    API_URL = "https://api.linkedin.com/v2/shares"
    HEADERS = {
        "Authorization": f"Bearer {LINKEDIN_ACCESS_TOKEN}",
        "Content-Type": "application/json",
        "X-Restli-Protocol-Version": "2.0.0"
    }
    
    # Construct the API payload
    payload = {
        "owner": organization_urn, 
        "subject": "Company Update",
        "text": {
            "text": text
        },
        "content": {
            "contentEntities": [],
            "shareMediaCategory": "NONE"
        },
        "distribution": {
            "linkedInDistributionTarget": {}
        }
    }

    print(f"\nAttempting to post to LinkedIn Organization: {organization_urn}")

    try:
        # Make the POST request
        response = requests.post(API_URL, headers=HEADERS, data=json.dumps(payload))
        
        # Handle the response
        if response.status_code == 201:
            share_urn = response.headers.get('x-restli-id')
            print("✅ Successfully posted an update to LinkedIn!")
            print(f"Share URN: {share_urn}")
            return share_urn
        else:
            error_details = response.json()
            print(f"❌ Post failed (Status: {response.status_code}): {error_details}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"❌ An error occurred during the network call: {e}")
        return None
    except Exception as e:
        print(f"❌ An unexpected error occurred: {e}")
        return None